In [9]:
from pynq import Overlay, MMIO
import time

conv_design = Overlay("./conv_design.bit")
conv_design.download()
print(conv_design.ip_dict.keys())


dict_keys(['axi_gpio_0', 'axi_gpio_1', 'axi_cdma_0', 'processing_system7_0'])


In [10]:
# === Address Mapping ===
CDMA_BASE   = 0x7E200000
DDR_SRC     = 0x20000000
DDR_DST     = 0x2000C000
BRAM0_ADDR  = 0xC0000000
BRAM1_ADDR  = 0xC2000000
GPIO0_ADDR  = 0x41200000
GPIO1_ADDR  = 0x41210000

in_bytes = 784 * 4
out_bytes = 676 * 4

# === MMIO ===
cdma     = MMIO(CDMA_BASE, 0x1000)
ddr_src  = MMIO(DDR_SRC, in_bytes)
ddr_dst  = MMIO(DDR_DST, in_bytes)
bram0    = MMIO(BRAM0_ADDR, 8000)
bram1    = MMIO(BRAM1_ADDR, 8000)
GPIO0    = MMIO(GPIO0_ADDR, 64)
GPIO1    = MMIO(GPIO1_ADDR, 64)


In [11]:
# === Step 1: Load HEX file to DDR memory ===
with open("input.hex", "r") as f:
    lines = f.readlines()

data = [int(line.strip(), 16) for line in lines]
byte_len = len(data) * 4

for i, val in enumerate(data):
    ddr_src.write(i * 4, val)

In [12]:
# === Step 2: DDR -> BRAM0 ===
cdma.write(0x00, 0x04)
cdma.write(0x18, DDR_SRC)
cdma.write(0x20, BRAM0_ADDR)
cdma.write(0x28, in_bytes)

# Wait for CDMA transfer to complete
while (cdma.read(0x04) & 0x2) == 0:
    pass

In [13]:
# --- Step 3: Send ready=1 via GPIO and wait for done=1 ---

GPIO0.write(0x4, 0x0)
GPIO1.write(0x4, 0x1)
GPIO0.write(0x0, 0x1)
while True:
    val = GPIO1.read(0x0)
    if val & 0x1:  # bit 1 = done
        break

In [14]:
# --- Step 4: BRAM1 -> DDR ---

cdma.write(0x00, 0x04)
cdma.write(0x18, BRAM1_ADDR)         # source: result from conv
cdma.write(0x20, DDR_DST)            # destination: back to DDR
cdma.write(0x28, out_bytes)          # result size
while (cdma.read(0x04) & 0x2) == 0:
    pass


In [15]:
# --- Step 5: Compare with golden.hex ---

with open("golden.hex", "r") as f:
    golden = [int(line.strip(), 16) for line in f]

success = True
print(f"\n Verifying {len(golden)} words against golden.hex:")
for i, expected in enumerate(golden):
    val = ddr_dst.read(i * 4)
    ok = (val == expected)
    print(f"Word {i:3}: 0x{val:08X} {'✅' if ok else '❌'} (expected: 0x{expected:08X})")
    if not ok:
        success = False

if success:
    print("\n Compare with golden.hex PASSED!")
else:
    print("\n Compare FAILED. Mismatch found.")



 Verifying 676 words against golden.hex:
Word   0: 0xFFFF9DF6 ✅ (expected: 0xFFFF9DF6)
Word   1: 0xFFFFA5C0 ✅ (expected: 0xFFFFA5C0)
Word   2: 0xFFFFACB7 ✅ (expected: 0xFFFFACB7)
Word   3: 0xFFFFB526 ✅ (expected: 0xFFFFB526)
Word   4: 0xFFFFBEE2 ✅ (expected: 0xFFFFBEE2)
Word   5: 0xFFFFC907 ✅ (expected: 0xFFFFC907)
Word   6: 0xFFFFD22F ✅ (expected: 0xFFFFD22F)
Word   7: 0xFFFFD711 ✅ (expected: 0xFFFFD711)
Word   8: 0xFFFFDCA3 ✅ (expected: 0xFFFFDCA3)
Word   9: 0xFFFFDD2B ✅ (expected: 0xFFFFDD2B)
Word  10: 0xFFFFDE10 ✅ (expected: 0xFFFFDE10)
Word  11: 0xFFFFDC83 ✅ (expected: 0xFFFFDC83)
Word  12: 0xFFFFDA71 ✅ (expected: 0xFFFFDA71)
Word  13: 0xFFFFD953 ✅ (expected: 0xFFFFD953)
Word  14: 0xFFFFD9F1 ✅ (expected: 0xFFFFD9F1)
Word  15: 0xFFFFDB21 ✅ (expected: 0xFFFFDB21)
Word  16: 0xFFFFDAB0 ✅ (expected: 0xFFFFDAB0)
Word  17: 0xFFFFDCBB ✅ (expected: 0xFFFFDCBB)
Word  18: 0xFFFFDDD4 ✅ (expected: 0xFFFFDDD4)
Word  19: 0xFFFFDFFB ✅ (expected: 0xFFFFDFFB)
Word  20: 0xFFFFE02E ✅ (expected: 0xFF